# Travel Planner Agent

In [1]:
from dotenv import load_dotenv
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
import os
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain.tools import tool,tool_node

load_dotenv()

True

### Initialization of LLm

In [6]:
llm_model = ChatGroq(
    model="openai/gpt-oss-20b"
)

### List of the Tools

In [7]:
# Search_tool
import requests
from langchain.tools import tool

web_search_tool = TavilySearch(
    max_results=5
)

@tool
def weather_tool(city:str)-> str:
    """
    Get the current weather of the city

    """
    weather_api_key = os.getenv("OPENWEATHER_API_KEY")

    url = (
        f"https://api.openweathermap.org/data/2.5/weather"
        f"?q={city}&appid={weather_api_key}&units=metric"
    )

    response = requests.get(url)

    weather_data = response.json()

    weather = weather_data["weather"][0]["description"]

    temp = weather_data["main"]["temp"]

    return (
        f"Weather in {city}: "
        f"{weather}, {temp}°C"
    )




In [8]:


class TravelState(TypedDict):
    destination: str
    budget: str
    days: int

    destination_info: str
    weather_info: str

    itinerary: str
    travel_tips: str

In [9]:
def destination_research_node(state: TravelState):

    destination = state["destination"]

    results = web_search_tool.invoke(
        f"best tourist attractions in {destination}"
    )

    return {
        "destination_info": str(results)
    }

In [10]:

def weather_node(state: TravelState):

    destination = state["destination"]

    weather = weather_tool.invoke(destination)

    return {
        "weather_info": weather
    }

In [11]:
def itinerary_node(state: TravelState):

    destination = state["destination"]

    budget = state["budget"]

    days = state["days"]

    info = state["destination_info"]

    weather = state["weather_info"]

    response = llm_model.invoke(
        f"""
        Create a {days}-day travel itinerary for:

        Destination: {destination}

        Budget: {budget}

        Destination Info:
        {info}

        Weather:
        {weather}

        Include:
        - places to visit
        - food suggestions
        - transportation
        - daily schedule
        - estimated costs
        """
    )

    return {
        "itinerary": response.content
    }

In [12]:
def tips_node(state: TravelState):

    destination = state["destination"]

    response = llm_model.invoke(
        f"""
        Give smart travel tips for visiting:

        {destination}

        Include:
        - safety tips
        - budgeting tips
        - local etiquette
        - best apps to use
        """
    )

    return {
        "travel_tips": response.content
    }

In [13]:

builder = StateGraph(TravelState)

In [14]:
builder.add_node(
    "destination_research",
    destination_research_node
)

builder.add_node(
    "weather",
    weather_node
)

builder.add_node(
    "itinerary",
    itinerary_node
)

builder.add_node(
    "tips",
    tips_node
)

In [15]:
builder.add_edge(
    START,
    "destination_research"
)

builder.add_edge(
    "destination_research",
    "weather"
)

builder.add_edge(
    "weather",
    "itinerary"
)

builder.add_edge(
    "itinerary",
    "tips"
)

builder.add_edge(
    "tips",
    END
)

In [17]:
graph = builder.compile()

In [18]:
result = graph.invoke({
    "destination": "Japan",
    "budget": "$2000",
    "days": 5
})

KeyError: 'weather'